# Qwen3-0.6B 明日方舟干员助手 LoRA 微调

基于 Qwen3-0.6B 模型，使用明日方舟干员数据集（449个干员，22621条问答）进行 LoRA 微调，打造一个能回答干员问题、扮演干员对话的专属助手。

**数据集**：[arknights-dataset](https://github.com/RainmeoX/arknights-dataset)

**环境**：AMD Radeon Cloud（ROCm 单卡）

## 1. 环境准备

In [ ]:
# 安装依赖
%pip install -q transformers datasets peft accelerate swanlab modelscope

In [ ]:
import torch
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

# 检查 GPU
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. 下载模型和数据集

In [ ]:
# 从 ModelScope 下载 Qwen3-0.6B
from modelscope import snapshot_download

model_path = snapshot_download('Qwen/Qwen3-0.6B', cache_dir='./models')
print('模型路径:', model_path)

In [ ]:
# 克隆数据集仓库（如果还没有）
import os
if not os.path.exists('arknights-dataset'):
    !git clone https://github.com/RainmeoX/arknights-dataset.git
else:
    print('数据集已存在')

In [ ]:
# 加载数据集
train_data = []
with open('arknights-dataset/data/train.jsonl', encoding='utf-8') as f:
    for line in f:
        train_data.append(eval(line))

print(f'训练集大小: {len(train_data)}')
print('示例:')
for item in train_data[:3]:
    print(f"  Q: {item['instruction']}")
    print(f"  A: {item['output'][:80]}")
    print()

## 3. 数据预处理

In [ ]:
# 转换为 Dataset 对象
df = pd.DataFrame(train_data)
ds = Dataset.from_pandas(df)
ds

In [ ]:
# 加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
tokenizer

In [ ]:
# 查看 chat_template 格式
messages = [
    {"role": "system", "content": "你是明日方舟游戏助手，可以回答关于干员的各种问题，也能扮演干员进行对话。"},
    {"role": "user", "content": "银灰是什么职业？"},
    {"role": "assistant", "content": "银灰是6星近卫干员，分支为领主。"},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
print(text)

In [ ]:
# 数据处理函数
SYSTEM_PROMPT = "你是明日方舟游戏助手，可以回答关于干员的各种问题，也能扮演干员进行对话。"

def process_func(example):
    MAX_LENGTH = 1024
    input_ids, attention_mask, labels = [], [], []
    
    # 构造 chat 格式
    instruction = tokenizer(
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n",
        add_special_tokens=False
    )
    response = tokenizer(f"{example['output']}<|im_end|>\n", add_special_tokens=False)
    
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    # instruction 部分不计算 loss
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
# 处理数据集
tokenized_id = ds.map(process_func, remove_columns=ds.column_names)
tokenized_id

In [ ]:
# 验证处理结果
print('=== 完整文本 ===')
print(tokenizer.decode(tokenized_id[0]['input_ids']))
print()
print('=== 标签部分（模型需要学习的） ===')
print(tokenizer.decode(list(filter(lambda x: x != -100, tokenized_id[0]["labels"]))))

## 4. 加载模型

In [ ]:
# 加载模型
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map="auto",
    torch_dtype=torch.bfloat16
)
model.enable_input_require_grads()  # 开启梯度检查点
print('模型 dtype:', model.dtype)

## 5. 配置 LoRA

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

## 6. 训练配置

In [ ]:
args = TrainingArguments(
    output_dir="./output/Qwen3_Arknights_LoRA",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=20,
    num_train_epochs=3,
    save_steps=200,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
)

In [ ]:
# SwanLab 训练监控
import swanlab
from swanlab.integration.transformers import SwanLabCallback

swanlab_callback = SwanLabCallback(
    project="Arknights-Finetune",
    experiment_name="Qwen3-0.6B-Arknights-LoRA"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_id,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[swanlab_callback]
)

## 7. 开始训练

In [ ]:
trainer.train()

## 8. 测试模型

In [ ]:
# 测试对话
def chat(prompt, system_prompt=SYSTEM_PROMPT):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False
    ).to(model.device)
    
    gen_kwargs = {"max_new_tokens": 512, "do_sample": True, "top_k": 50, "top_p": 0.9, "temperature": 0.7}
    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)
        outputs = outputs[:, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 测试几个问题
test_questions = [
    "银灰是什么职业的干员？",
    "扮演阿米娅，跟我打个招呼",
    "推荐几个强力的狙击干员",
    "真银斩是什么技能？",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"问：{q}")
    print(f"答：{chat(q)}")

## 9. 保存模型

In [ ]:
# 保存 LoRA adapter
model.save_pretrained("./output/Qwen3_Arknights_LoRA_final")
tokenizer.save_pretrained("./output/Qwen3_Arknights_LoRA_final")
print('模型已保存到 ./output/Qwen3_Arknights_LoRA_final')

## 10. 重新加载模型推理

In [ ]:
from peft import PeftModel

# 重新加载基础模型
base_model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map="auto",
    torch_dtype=torch.bfloat16, 
    trust_remote_code=True
)

# 加载 LoRA 权重
model = PeftModel.from_pretrained(base_model, model_id="./output/Qwen3_Arknights_LoRA_final")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

print('模型加载完成')

In [ ]:
# 最终测试
questions = [
    "介绍一下银灰这名干员",
    "扮演银灰，说一句任命助理的台词",
    "6星医疗干员有哪些？",
    "星熊的满级属性是多少？",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"问：{q}")
    print(f"答：{chat(q)}")

## 11. 运行评估（可选）

训练完成后，可以运行评估脚本测试模型效果：

```bash
# 1. 生成预测结果
python arknights-dataset/eval/generate_predictions.py \
    --model_path ./output/Qwen3_Arknights_LoRA_final \
    --base_model_path {model_path} \
    --test_dir arknights-dataset/eval/ \
    --output predictions.json

# 2. 运行评估
python arknights-dataset/eval/evaluate.py \
    --predictions predictions.json \
    --test_dir arknights-dataset/eval/ \
    --output eval_report.json \
    --fluency_score 85
```